In [151]:
import pandas as pd
from pathlib import Path

dossier_data = Path('DATA/')  # chemin réel des fichiers sources

rh_absences = pd.read_csv('DATA/rh_absences.csv')
rh_departures = pd.read_csv('DATA/rh_departures.csv')
rh_employees = pd.read_csv('DATA/rh_employees.csv')

fichiers = [
    ('rh_absences', rh_absences),
    ('rh_departures', rh_departures),
    ('rh_employees', rh_employees),
]

# ============================================================
# 1. VIEW DATA
# ============================================================

In [152]:
# Afficher les 5 premières lignes des abscences
rh_absences.head(5)

,absence_id,employee_id,annee,mois,date_debut,date_fin,nb_jours,motif,justifiee,departement
0,13323,2879,2022,1,2022-01-19,2022-01-26,8,Congé payé,Oui,Finance
1,15714,2012,2023,5,2023-05-02,2023-05-07,6,Maladie,Oui,R&D
2,10557,1177,2023,5,2023-05-12,2023-05-28,17,Absence injustifiée,Non,RH
3,11946,2445,2023,8,2023-08-27,2023-08-28,2,Maladie,Oui,Logistique
4,12442,1496,2024,8,2024-08-25,2024-09-10,17,sick leave,Oui,Finance


In [153]:
# Afficher les 5 premières lignes des départs
rh_departures.head(5)

,depart_id,employee_id,date_depart,motif_depart,duree_preavis_jours,anciennete_au_depart,departement,contrat,entretien_sortie,satisfaction_score,recommanderait_employeur,poste_pourvu
0,5000,3153,2016-05-13,NaN,0,-0.7,Direction,CDD,Non,NaN,Non,Oui
1,5001,2343,2022-10-24,NaN,90,-0.4,Marketing,CDI,Non,NaN,Oui,Oui
2,5002,2765,2017-03-05,NaN,90,-0.3,RH,CDI,Non,3.0,Non,Non
3,5003,1416,2024-01-15,NaN,0,-0.3,R&D,Alternance,Non,4.0,NaN,NaN
4,5004,2165,2023-01-18,NaN,90,-0.9,RH,Alternance,Non,NaN,Oui,NaN


In [154]:
# Afficher les 5 premières lignes des employés

rh_employees.head(5)

,employee_id,nom,prenom,genre,date_naissance,email,departement,ville,contrat,date_embauche,anciennete_annees,salaire_mensuel_brut,manager_id,temps_partiel,taux_activite
0,2525,David,Louis,F,2014-08-31,louis.daviddatascope.fr,Operations,Nice,CDD,2022-11-15,2.1,0.0,2515.0,Non,100
1,2502,Leroy,Inès,F,1970-05-31,inès.leroy@datascope.fr,RH,Grenoble,CDI,2020-11-07,4.1,2326.0,2525.0,Non,100
2,2036,Robert,Antoine,M,1975-08-06,antoine.robert@datascope.fr,Informatique,Nice,CDI,2024-08-14,0.4,2888.0,1561.0,Non,80
3,1450,Dubois,Marie,M,1977-07-15,marie.dubois@datascope.fr,Finance,Strasbourg,CDI,2020-05-28,4.6,3074.0,2515.0,Non,100
4,3362,Roux,Léa,M,1980-06-23,léa.roux@datascope.fr,Marketing,Lyon,CDI,2023-10-29,1.2,3696.0,2630.0,Non,50


# ============================================================
# 2. VOLUMÉTRIE
# ============================================================

In [155]:
resultats = []  # liste qui va stocker un dict par fichier

# Boucle sur les fichiers pour calculer la volumétrie
for nom, df in fichiers:
    # reconstruit le chemin du CSV source pour lire sa taille sur disque
    chemin = dossier_data / f'{nom}.csv'
    resultats.append({
        'fichier': nom,
        # nb de lignes du DataFrame déjà chargé
        'lignes': df.shape[0],
        # nb de colonnes du DataFrame déjà chargé
        'colonnes': df.shape[1],
        # taille du fichier brut sur disque (octets -> Mo)
        'taille_disque_Mo': round(chemin.stat().st_size / 1e6, 2),
        # taille réelle en RAM (deep=True inclut les objets str)
        'taille_memoire_Mo': round(df.memory_usage(deep=True).sum() / 1e6, 2),
    })

print('=== 2. VOLUMÉTRIE ===')
# tableau récap, une ligne par fichier
display(pd.DataFrame(resultats).set_index('fichier'))

=== 2. VOLUMÉTRIE ===


,lignes,colonnes,taille_disque_Mo,taille_memoire_Mo
fichier,,,,
rh_absences,6140,10,0.43,2.05
rh_departures,380,12,0.03,0.16
rh_employees,2447,15,0.28,1.51


# ============================================================
# 3. QUALITÉ
# ============================================================

In [156]:
resultats = []

for nom, df in fichiers:
    resultats.append({
        'fichier': nom,
        'doublons_lignes': df.duplicated().sum(),
        'cellules_vides_%': round(df.isna().sum().sum() / df.size * 100, 2)
    })

print('\n=== 2. QUALITÉ (global) ===')
display(pd.DataFrame(resultats).set_index('fichier'))




=== 2. QUALITÉ (global) ===


,doublons_lignes,cellules_vides_%
fichier,,
rh_absences,0,0.00
rh_departures,0,7.39
rh_employees,47,1.49


## RH Employee

In [157]:
rh_employees[rh_employees.duplicated(keep=False)].sort_values(by=list(rh_employees.columns))

,employee_id,nom,prenom,genre,date_naissance,email,departement,ville,contrat,date_embauche,anciennete_annees,salaire_mensuel_brut,manager_id,temps_partiel,taux_activite
711,1020,Thomas,Chloé,F,1976-02-28,chloé.thomas@datascope.fr,Juridique,Nice,CDI,2020-03-16,4.8,2808.0,3284.0,Oui,80
928,1020,Thomas,Chloé,F,1976-02-28,chloé.thomas@datascope.fr,Juridique,Nice,CDI,2020-03-16,4.8,2808.0,3284.0,Oui,80
909,1029,Dubois,Camille,F,1996-05-16,camille.dubois@datascope.fr,Finance,Nice,CDI,2021-10-09,3.2,3224.0,2645.0,Non,50
2427,1029,Dubois,Camille,F,1996-05-16,camille.dubois@datascope.fr,Finance,Nice,CDI,2021-10-09,3.2,3224.0,2645.0,Non,50
1592,1052,Petit,Camille,M,1969-10-15,camille.petit@datascope.fr,Finance,Strasbourg,CDI,2023-05-01,1.7,2382.0,1747.0,Non,50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1194,3264,Simon,Paul,F,1984-08-25,paul.simon@datascope.fr,R&D,Strasbourg,CDD,2016-11-30,8.1,1261.0,2630.0,Non,50
356,3341,Durand,Romain,F,1966-01-10,romain.durand@datascope.fr,RH,Marseille,CDI,2018-08-24,6.4,3159.0,1201.0,Non,100
627,3341,Durand,Romain,F,1966-01-10,romain.durand@datascope.fr,RH,Marseille,CDI,2018-08-24,6.4,3159.0,1201.0,Non,100
1726,3368,Robert,Marie,M,1974-10-27,marie.robert@datascope.fr,Informatique,Strasbourg,Alternance,2020-08-09,4.4,1054.0,1948.0,Non,50


**Observation** : Suprimer les lignes en doublons

In [158]:
detail = pd.DataFrame({
    'type': rh_employees.dtypes,
    'nulls': rh_employees.isna().sum(),
    'nulls_%': (rh_employees.isna().mean() * 100).round(2),
    'uniques': rh_employees.nunique(),
})
display(detail)

,type,nulls,nulls_%,uniques
employee_id,int64,0,0.00,2400
nom,str,0,0.00,20
prenom,str,0,0.00,20
genre,str,0,0.00,3
date_naissance,str,0,0.00,2208
email,str,0,0.00,431
departement,str,0,0.00,17
ville,str,0,0.00,10
contrat,str,0,0.00,5
date_embauche,str,0,0.00,1758


In [159]:
rh_employees.head()

,employee_id,nom,prenom,genre,date_naissance,email,departement,ville,contrat,date_embauche,anciennete_annees,salaire_mensuel_brut,manager_id,temps_partiel,taux_activite
0,2525,David,Louis,F,2014-08-31,louis.daviddatascope.fr,Operations,Nice,CDD,2022-11-15,2.1,0.0,2515.0,Non,100
1,2502,Leroy,Inès,F,1970-05-31,inès.leroy@datascope.fr,RH,Grenoble,CDI,2020-11-07,4.1,2326.0,2525.0,Non,100
2,2036,Robert,Antoine,M,1975-08-06,antoine.robert@datascope.fr,Informatique,Nice,CDI,2024-08-14,0.4,2888.0,1561.0,Non,80
3,1450,Dubois,Marie,M,1977-07-15,marie.dubois@datascope.fr,Finance,Strasbourg,CDI,2020-05-28,4.6,3074.0,2515.0,Non,100
4,3362,Roux,Léa,M,1980-06-23,léa.roux@datascope.fr,Marketing,Lyon,CDI,2023-10-29,1.2,3696.0,2630.0,Non,50


In [160]:
# Analyse les salaires moyen à 0 ou vide par type de contrat
rh_employees[(rh_employees['salaire_mensuel_brut'] == 0) | (rh_employees['salaire_mensuel_brut'].isna())].groupby('contrat').size()

contrat
Alternance    13
CDD           26
CDI           69
Interim        4
Stage         10
dtype: int64

In [161]:
rh_employees[rh_employees['salaire_mensuel_brut'] > 0].groupby(
    ['departement', 'ville', 'contrat', 'taux_activite']
)['salaire_mensuel_brut'].mean().round(2)

departement  ville     contrat     taux_activite
Commercial   Bordeaux  Alternance  50               1408.00
                                   100               947.00
                       CDD         80               1622.00
                                   100              2393.00
                       CDI         50               3249.00
                                                     ...   
RH           Toulouse  CDD         100              2429.75
                       CDI         80               2320.00
                                   100              2710.40
                       Interim     100              2078.00
                       Stage       100               900.00
Name: salaire_mensuel_brut, Length: 822, dtype: float64

In [162]:
rh_employees[(rh_employees['contrat'] == "CDI") & (rh_employees['temps_partiel'] == "Oui")]

,employee_id,nom,prenom,genre,date_naissance,email,departement,ville,contrat,date_embauche,anciennete_annees,salaire_mensuel_brut,manager_id,temps_partiel,taux_activite
6,2324,Lefebvre,Alice,F,1984-06-24,alice.lefebvre@datascope.fr,Commercial,Paris,CDI,2021-09-25,3.3,2169.0,1005.0,Oui,50
16,1256,Martin,Inès,M,1992-05-03,inès.martin@datascope.fr,Logistique,Toulouse,CDI,2018-01-12,7.0,1998.0,2619.0,Oui,100
21,1769,Laurent,Léa,M,1967-12-27,léa.laurent@datascope.fr,Opérations,Toulouse,CDI,2015-08-13,9.4,2682.0,1195.0,Oui,80
27,1281,Petit,Sophie,F,1988-02-05,sophie.petit@datascope.fr,Direction,Grenoble,CDI,2015-02-15,9.9,1813.0,3378.0,Oui,100
30,1673,David,Julie,F,1978-05-12,julie.david@datascope.fr,Direction,Paris,CDI,2016-02-27,8.8,1361.0,NaN,Oui,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2431,1262,Petit,Théo,M,1971-06-07,théo.petit@datascope.fr,RH,Marseille,CDI,2020-02-21,4.9,3060.0,2122.0,Oui,100
2436,2418,Durand,Emma,F,1970-01-10,emma.durand@datascope.fr,R&D,Lille,CDI,2021-03-01,3.8,3455.0,2122.0,Oui,100
2440,1192,Bertrand,Emma,M,1973-07-08,emma.bertrand@datascope.fr,Direction,Nice,CDI,2021-10-13,3.2,3389.0,1569.0,Oui,80
2442,2266,Lefebvre,Julie,F,1966-03-17,julie.lefebvre@datascope.fr,Marketing,Paris,CDI,2023-08-01,1.4,1731.0,1127.0,Oui,100


In [ ]:
# ============================================================
# 3. ANOMALIES — rh_employees
# ============================================================
print('\n=== 3. ANOMALIES : rh_employees ===')

# --- Numériques : négatifs et outliers (méthode IQR) ---
num = rh_employees.select_dtypes(include='number')  # sélectionne les colonnes numériques uniquement
if not num.empty:
    q1, q3 = num.quantile(0.25), num.quantile(0.75)  # 1er et 3e quartile, par colonne
    iqr = q3 - q1  # écart interquartile
    anomalies_num = pd.DataFrame({
        'min': num.min(),
        'max': num.max(),
        'negatifs': (num < 0).sum(),  # nb de valeurs négatives (incohérent pour ancienneté, salaire...)
        'outliers_IQR': ((num < q1 - 1.5 * iqr) | (num > q3 + 1.5 * iqr)).sum(),  # hors bornes IQR classiques
    })
    print('Numériques :')
    display(anomalies_num)

# --- Texte : espaces parasites, variantes de casse, chaînes vides ---
txt = rh_employees.select_dtypes(include='object')  # sélectionne les colonnes texte
if not txt.empty:
    anomalies_txt = pd.DataFrame({
        # compare la valeur brute à sa version strip() : différence = espace en début/fin
        'espaces_parasites': [(txt[c].dropna() != txt[c].dropna().str.strip()).sum() for c in txt],
        # chaîne non-nulle mais vide une fois les espaces retirés
        'chaines_vides': [(txt[c].dropna().str.strip() == '').sum() for c in txt],
        # écart entre nb de valeurs uniques brutes et nb de valeurs uniques normalisées (casse/espaces)
        'variantes_casse': [txt[c].nunique() - txt[c].str.lower().str.strip().nunique() for c in txt],
    }, index=txt.columns)
    print('Texte :')
    display(anomalies_txt)

# --- Dates : colonnes dont le nom contient "date" ---
cols_date = [c for c in rh_employees.columns if 'date' in c.lower()]  # repère les colonnes dates par leur nom
if cols_date:
    lignes = []
    for c in cols_date:
        d = pd.to_datetime(rh_employees[c], errors='coerce')  # conversion, NaT si format invalide
        lignes.append({
            'colonne': c,
            # invalides = valeurs devenues NaT à la conversion, MAIS qui n'étaient pas déjà NaN à la base
            'invalides': d.isna().sum() - rh_employees[c].isna().sum(),
            'min': d.min(),
            'max': d.max(),
            'dans_le_futur': (d > pd.Timestamp.today()).sum(),  # date postérieure à aujourd'hui = suspect
        })
    print('Dates :')
    display(pd.DataFrame(lignes).set_index('colonne'))


=== 3. ANOMALIES : rh_employees ===
Numériques :


,min,max,negatifs,outliers_IQR
employee_id,1001.0,3400.0,0,0
anciennete_annees,-2.5,10.0,15,0
salaire_mensuel_brut,0.0,4715.0,0,0
manager_id,1005.0,3378.0,0,0
taux_activite,50.0,100.0,0,0


Texte :


C:\Users\blanc\AppData\Local\Temp\ipykernel_11248\2513653118.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  txt = rh_employees.select_dtypes(include='object')  # sélectionne les colonnes texte


,espaces_parasites,chaines_vides,variantes_casse
nom,0,0,0
prenom,0,0,0
genre,0,0,0
date_naissance,0,0,0
email,0,0,0
departement,0,0,0
ville,0,0,0
contrat,0,0,0
date_embauche,0,0,0
temps_partiel,0,0,0


Dates :


,invalides,min,max,dans_le_futur
colonne,,,,
date_naissance,0,1965-01-01,2015-11-05,0
date_embauche,0,2015-01-03,2024-12-29,0


In [164]:
rh_employees[rh_employees['anciennete_annees'] < 0 ]

,employee_id,nom,prenom,genre,date_naissance,email,departement,ville,contrat,date_embauche,anciennete_annees,salaire_mensuel_brut,manager_id,temps_partiel,taux_activite
321,1880,Vincent,Pierre,F,2013-03-31,pierre.vincentdatascope.fr,Operations,Strasbourg,CDI,2024-10-31,-0.2,0.0,NaN,Non,100
385,2607,David,Romain,M,2012-04-15,romain.daviddatascope.fr,Operations,Lyon,CDI,2019-08-08,-1.3,0.0,2630.0,Non,50
442,1875,Vincent,Pierre,F,2012-02-02,pierre.vincentdatascope.fr,IT,Bordeaux,CDD,2015-02-27,-1.9,0.0,NaN,Oui,80
803,3004,Bernard,Antoine,M,2014-04-07,antoine.bernarddatascope.fr,Sales,Strasbourg,CDI,2016-08-28,-2.1,0.0,1005.0,Non,100
973,1641,Durand,Inès,F,2012-09-07,inès.duranddatascope.fr,Logistics,Bordeaux,CDI,2017-09-20,-0.2,0.0,NaN,Oui,100
1101,3378,Petit,Paul,F,2013-11-19,paul.petitdatascope.fr,Sales,Bordeaux,CDI,2019-12-17,-1.7,0.0,3028.0,Non,100
1378,1196,Laurent,Pierre,F,2013-06-27,pierre.laurentdatascope.fr,Logistics,Strasbourg,CDI,2022-09-28,-0.4,0.0,2461.0,Oui,100
1404,1519,Garcia,Inès,F,2015-07-17,inès.garciadatascope.fr,Operations,Toulouse,Interim,2024-12-21,-2.4,0.0,NaN,Non,100
1447,1790,Simon,Paul,M,2011-01-26,paul.simondatascope.fr,Legal,Bordeaux,CDI,2017-01-26,-2.5,0.0,NaN,Oui,100
1730,2619,Roux,Marie,F,2014-03-24,marie.rouxdatascope.fr,Marketing,Nantes,Interim,2018-07-09,-2.4,0.0,1092.0,Non,50


**Observation** : 

* Choisir la formule pour remplacer les salaires mensuel : Median par groupe 
* Les valeur null ou 0 dans les stage peuvent être normal du coup je ne pas modifier ou mettre à 0 les valeurs

* Suprimer et ajouter la colonne temp_partiel correctement en fonction du taux d'activite
* Retirer la colonne manager ID
* Calculer l'ages + faire des groupes d'ages
* Anomysé lles données : retirer colonne Nom, prénom date de naissance, age( 31/12/2024) garder le groupe d'ages.


**Exploration** : 

* Vérifier les données de date d'ancienneté entre date d'embauche et date de départ

## Rh Absences

In [165]:
detail = pd.DataFrame({
    'type': rh_absences.dtypes,
    'nulls': rh_absences.isna().sum(),
    'nulls_%': (rh_absences.isna().mean() * 100).round(2),
    'uniques': rh_absences.nunique(),
})
display(detail)

,type,nulls,nulls_%,uniques
absence_id,int64,0,0.0,6140
employee_id,int64,0,0.0,400
annee,int64,0,0.0,3
mois,int64,0,0.0,12
date_debut,str,0,0.0,1000
date_fin,str,0,0.0,1105
nb_jours,int64,0,0.0,42
motif,str,0,0.0,11
justifiee,str,0,0.0,2
departement,str,0,0.0,14


In [ ]:
# ============================================================
# 3. ANOMALIES — rh_employees
# ============================================================
print('\n=== 3. ANOMALIES : rh_employees ===')

# --- Numériques : négatifs et outliers (méthode IQR) ---
num = rh_absences.select_dtypes(include='number')  # sélectionne les colonnes numériques uniquement
if not num.empty:
    q1, q3 = num.quantile(0.25), num.quantile(0.75)  # 1er et 3e quartile, par colonne
    iqr = q3 - q1  # écart interquartile
    anomalies_num = pd.DataFrame({
        'min': num.min(),
        'max': num.max(),
        'negatifs': (num < 0).sum(),  # nb de valeurs négatives (incohérent pour ancienneté, salaire...)
        'outliers_IQR': ((num < q1 - 1.5 * iqr) | (num > q3 + 1.5 * iqr)).sum(),  # hors bornes IQR classiques
    })
    print('Numériques :')
    display(anomalies_num)

# --- Texte : espaces parasites, variantes de casse, chaînes vides ---
txt = rh_absences.select_dtypes(include='object')  # sélectionne les colonnes texte
if not txt.empty:
    anomalies_txt = pd.DataFrame({
        # compare la valeur brute à sa version strip() : différence = espace en début/fin
        'espaces_parasites': [(txt[c].dropna() != txt[c].dropna().str.strip()).sum() for c in txt],
        # chaîne non-nulle mais vide une fois les espaces retirés
        'chaines_vides': [(txt[c].dropna().str.strip() == '').sum() for c in txt],
        # écart entre nb de valeurs uniques brutes et nb de valeurs uniques normalisées (casse/espaces)
        'variantes_casse': [txt[c].nunique() - txt[c].str.lower().str.strip().nunique() for c in txt],
    }, index=txt.columns)
    print('Texte :')
    display(anomalies_txt)

# --- Dates : colonnes dont le nom contient "date" ---
cols_date = [c for c in rh_absences.columns if 'date' in c.lower()]  # repère les colonnes dates par leur nom
if cols_date:
    lignes = []
    for c in cols_date:
        d = pd.to_datetime(rh_absences[c], errors='coerce')  # conversion, NaT si format invalide
        lignes.append({
            'colonne': c,
            # invalides = valeurs devenues NaT à la conversion, MAIS qui n'étaient pas déjà NaN à la base
            'invalides': d.isna().sum() - rh_employees[c].isna().sum(),
            'min': d.min(),
            'max': d.max(),
            'dans_le_futur': (d > pd.Timestamp.today()).sum(),  # date postérieure à aujourd'hui = suspect
        })
    print('Dates :')
    display(pd.DataFrame(lignes).set_index('colonne'))


=== 3. ANOMALIES : rh_employees ===
Numériques :


,min,max,negatifs,outliers_IQR
absence_id,10000,16139,0,0
employee_id,1015,3398,0,0
annee,2022,2024,0,0
mois,1,12,0,0
nb_jours,-21,22,91,19


Texte :


C:\Users\blanc\AppData\Local\Temp\ipykernel_11248\2340361326.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  txt = rh_absences.select_dtypes(include='object')  # sélectionne les colonnes texte


,espaces_parasites,chaines_vides,variantes_casse
date_debut,0,0,0
date_fin,0,0,0
motif,0,0,0
justifiee,0,0,0
departement,0,0,0


Dates :


,invalides,min,max,dans_le_futur
colonne,,,,
date_debut,0,2022-01-01,2024-12-28,0
date_fin,0,2022-01-01,2025-01-17,0


In [167]:
rh_absences[rh_absences['nb_jours'] < 0]

,absence_id,employee_id,annee,mois,date_debut,date_fin,nb_jours,motif,justifiee,departement
99,10079,2193,2023,10,2023-10-24,2023-10-24,-1,Enfant malade,Oui,Marketing
234,10046,2573,2022,12,2022-12-28,2022-12-21,-8,Congé payé,Oui,RH
262,16137,1596,2022,3,2022-03-25,2022-03-29,-4,paid leave,Oui,R&D
291,10072,2712,2022,1,2022-01-16,2022-01-17,-2,RTT,Oui,Logistique
368,10043,1212,2022,1,2022-01-04,2022-01-01,-4,Congé sans solde,Oui,R&D
...,...,...,...,...,...,...,...,...,...,...
5877,10055,1421,2023,3,2023-03-28,2023-04-08,-12,Congé sans solde,Oui,Finance
5887,10008,2930,2024,11,2024-11-16,2024-11-12,-5,Accident du travail,Oui,Logistique
5923,10018,1301,2023,6,2023-06-20,2023-06-10,-11,RTT,Oui,Commercial
6056,10063,3371,2024,5,2024-05-02,2024-05-09,-8,RTT,Oui,Direction


In [177]:
rh_absences.info()

<class 'pandas.DataFrame'>
RangeIndex: 6140 entries, 0 to 6139
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   absence_id   6140 non-null   int64
 1   employee_id  6140 non-null   int64
 2   annee        6140 non-null   int64
 3   mois         6140 non-null   int64
 4   date_debut   6140 non-null   str  
 5   date_fin     6140 non-null   str  
 6   nb_jours     6140 non-null   int64
 7   motif        6140 non-null   str  
 8   justifiee    6140 non-null   str  
 9   departement  6140 non-null   str  
dtypes: int64(5), str(5)
memory usage: 479.8 KB


In [181]:
rh_absences[rh_absences['date_debut'] > rh_absences['date_fin']]

,absence_id,employee_id,annee,mois,date_debut,date_fin,nb_jours,motif,justifiee,departement
234,10046,2573,2022,12,2022-12-28,2022-12-21,-8,Congé payé,Oui,RH
368,10043,1212,2022,1,2022-01-04,2022-01-01,-4,Congé sans solde,Oui,R&D
393,10051,1232,2024,6,2024-06-21,2024-06-15,-7,Maternité/Paternité,Oui,Marketing
483,10003,2153,2022,6,2022-06-20,2022-06-17,-4,Maladie,Oui,Commercial
536,10006,2333,2023,9,2023-09-04,2023-08-25,-11,RTT,Oui,Marketing
611,10010,2971,2023,8,2023-08-28,2023-08-22,-7,Formation,Oui,Opérations
779,10004,2136,2024,5,2024-05-27,2024-05-25,-3,Maladie,Oui,Juridique
800,10050,2040,2023,10,2023-10-15,2023-10-11,-5,Congé payé,Oui,Opérations
801,10009,3356,2022,7,2022-07-03,2022-06-28,-6,Formation,Oui,Finance
937,10036,2036,2024,6,2024-06-05,2024-05-29,-8,Maladie,Oui,Informatique


**Observation** : 

* Permuter la date de début et la date de fin si la date de début > Date de fin
* Recalculer le nombre de jours


## RH Departures

In [168]:
detail = pd.DataFrame({
    'type': rh_departures.dtypes,
    'nulls': rh_departures.isna().sum(),
    'nulls_%': (rh_departures.isna().mean() * 100).round(2),
    'uniques': rh_departures.nunique(),
})
display(detail)

,type,nulls,nulls_%,uniques
depart_id,int64,0,0.00,380
employee_id,int64,0,0.00,228
date_depart,str,0,0.00,265
motif_depart,str,18,4.74,9
duree_preavis_jours,int64,0,0.00,4
anciennete_au_depart,float64,0,0.00,74
departement,str,0,0.00,13
contrat,str,0,0.00,6
entretien_sortie,str,0,0.00,2
satisfaction_score,float64,100,26.32,5


In [169]:
pd.crosstab(rh_departures['entretien_sortie'], rh_departures['satisfaction_score'].isna())

satisfaction_score,False,True
entretien_sortie,,
Non,149,55
Oui,131,45


In [174]:
# ============================================================
# 3. ANOMALIES — rh_employees
# ============================================================
print('\n=== 3. ANOMALIES : rh_departures ===')

# --- Numériques : négatifs et outliers (méthode IQR) ---
num = rh_departures.select_dtypes(include='number')  # sélectionne les colonnes numériques uniquement
if not num.empty:
    q1, q3 = num.quantile(0.25), num.quantile(0.75)  # 1er et 3e quartile, par colonne
    iqr = q3 - q1  # écart interquartile
    anomalies_num = pd.DataFrame({
        'min': num.min(),
        'max': num.max(),
        'negatifs': (num < 0).sum(),  # nb de valeurs négatives (incohérent pour ancienneté, salaire...)
        'outliers_IQR': ((num < q1 - 1.5 * iqr) | (num > q3 + 1.5 * iqr)).sum(),  # hors bornes IQR classiques
    })
    print('Numériques :')
    display(anomalies_num)

# --- Texte : espaces parasites, variantes de casse, chaînes vides ---
txt = rh_departures.select_dtypes(include='object')  # sélectionne les colonnes texte
if not txt.empty:
    anomalies_txt = pd.DataFrame({
        # compare la valeur brute à sa version strip() : différence = espace en début/fin
        'espaces_parasites': [(txt[c].dropna() != txt[c].dropna().str.strip()).sum() for c in txt],
        # chaîne non-nulle mais vide une fois les espaces retirés
        'chaines_vides': [(txt[c].dropna().str.strip() == '').sum() for c in txt],
        # écart entre nb de valeurs uniques brutes et nb de valeurs uniques normalisées (casse/espaces)
        'variantes_casse': [txt[c].nunique() - txt[c].str.lower().str.strip().nunique() for c in txt],
    }, index=txt.columns)
    print('Texte :')
    display(anomalies_txt)

# --- Dates : colonnes dont le nom contient "date" ---
cols_date = [c for c in rh_departures.columns if 'date' in c.lower()]  # repère les colonnes dates par leur nom
if cols_date:
    lignes = []
    for c in cols_date:
        d = pd.to_datetime(rh_departures[c], errors='coerce')  # conversion, NaT si format invalide
        lignes.append({
            'colonne': c,
            # invalides = valeurs devenues NaT à la conversion, MAIS qui n'étaient pas déjà NaN à la base
            'invalides': d.isna().sum() - rh_departures[c].isna().sum(),
            'min': d.min(),
            'max': d.max(),
            'dans_le_futur': (d > pd.Timestamp.today()).sum(),  # date postérieure à aujourd'hui = suspect
        })
    print('Dates :')
    display(pd.DataFrame(lignes).set_index('colonne'))


=== 3. ANOMALIES : rh_departures ===
Numériques :


,min,max,negatifs,outliers_IQR
depart_id,5000.0,5379.0,0,0
employee_id,1015.0,9082.0,0,12
duree_preavis_jours,0.0,90.0,0,0
anciennete_au_depart,-0.9,6.8,22,0
satisfaction_score,1.0,5.0,0,0


Texte :


C:\Users\blanc\AppData\Local\Temp\ipykernel_11248\2682725070.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  txt = rh_departures.select_dtypes(include='object')  # sélectionne les colonnes texte


,espaces_parasites,chaines_vides,variantes_casse
date_depart,0,0,0
motif_depart,0,0,0
departement,0,0,0
contrat,0,0,0
entretien_sortie,0,0,0
recommanderait_employeur,0,0,0
poste_pourvu,0,0,0


Dates :


,invalides,min,max,dans_le_futur
colonne,,,,
date_depart,0,2015-06-06,2024-12-31,0


In [175]:
rh_departures[rh_departures['date_depart'] < '2022-01-01'][['depart_id', 'employee_id', 'date_depart']]

,depart_id,employee_id,date_depart
0,5000,3153,2016-05-13
2,5002,2765,2017-03-05
5,5005,1631,2017-10-12
7,5007,2743,2020-07-25
9,5009,3056,2017-02-26
...,...,...,...
365,5365,1927,2018-03-25
366,5366,3189,2020-08-14
369,5369,9032,2020-06-05
370,5370,9082,2020-03-23


**Observation** : 

* Recalculer le nombre de jour 
* Je ne sais pas par quoi remplacer la satisfaction score (Ne pas emputer)